In [1]:
from huggingface_hub import notebook_login
from transformers import AutoModelForCausalLM, AutoConfig, TrainingArguments, Trainer
import math
from datasets import load_dataset
from transformers import AutoTokenizer

**Squad Dataset Training**

In [ ]:
#loading the SQuaD Dataset
sq_dataset = load_dataset("squad")
sq_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [ ]:
# function to add the end of text
def add_end_of_text_to_question(example):
  example["question"] = example["question"] + " <|end_of_text|>"
  return example


sq_dataset = sq_dataset.remove_columns(['id', 'title', 'context', 'answers'])
sq_dataset = sq_dataset.map(add_end_of_text_to_question)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
 # Define which model we're using — this determines the tokenizer rules
model_checkpoint = "distilgpt2"

# Load the same tokenizer that was used when DistilGPT-2 was originally trained
# use_fast=True uses the Rust-based fast tokenizer for speed
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

# Define the tokenization function — converts raw question text into token IDs
# truncation=True ensures any question longer than the model's max length gets cut off
def tokenize_functions(examples):
    return tokenizer(examples["question"], truncation=True)

# Apply the tokenization function across the entire dataset
# batched=True processes multiple examples at once for efficiency
# num_proc=4 uses 4 CPU cores in parallel to speed things up
# remove_columns=['question'] drops the original text column — we no longer need it
# since it's now been replaced by input_ids and attention_mask
sq_tokenized_datasets = sq_dataset.map(
    tokenize_functions,
    batched=True,
    num_proc=4,
    remove_columns=['question']
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
# Each chunk will be exactly 128 tokens long
block_size = 128

def group_texts(examples):
    # Step 1 — Concatenate all token sequences into one long stream
    # sum(examples[k], []) flattens a list of lists into a single list
    # e.g. [[1,2,3], [4,5,6]] becomes [1,2,3,4,5,6]
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}

    # Step 2 — Get the total length of the concatenated stream
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    # Step 3 — Trim the total length to be divisible by block_size
    # This drops any leftover tokens at the end that don't fill a complete chunk
    # e.g. if total is 1050 and block_size is 128, total becomes 1024 (8 full chunks)
    total_length = (total_length // block_size) * block_size

    # Step 4 — Slice the long stream into equal 128-token chunks
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

    # Step 5 — Copy input_ids as labels
    # The model's task is to predict the next token at every position
    # so the label at each position is just the token that follows it
    # The Transformers library handles the one-position shift automatically
    result["labels"] = result["input_ids"].copy()
    return result

# Apply group_texts across the full tokenized dataset
# batch_size=1000 means it processes 1000 examples at a time before chunking
# num_proc=4 runs across 4 CPU cores in parallel
sq_lm_datasets = sq_tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4
)

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
# Shrink dataset for faster demo
small_sq_train_dataset = sq_lm_datasets["train"].shuffle(seed=42).select(range(100))
small_sq_eval_dataset = sq_lm_datasets["validation"].shuffle(seed=42).select(range(100))

Fetching model blueprint...
Building a blank model from scratch...


In [ ]:
# Log into Hugging Face Hub so we can push the model after training
notebook_login()

# Step A: Get the blueprint for the model
# AutoConfig loads only the architecture definition of DistilGPT-2
# — the number of layers, attention heads, hidden size etc.
# — but none of the learned weights
print("Fetching model blueprint...")
config = AutoConfig.from_pretrained(model_checkpoint)

# Step B: Build a blank model using that blueprint
# from_config() creates the model structure with completely random weights
# This model has zero knowledge of language at this point
print("Building a blank model from scratch...")
model = AutoModelForCausalLM.from_config(config)

# Define the training configuration
training_arg = TrainingArguments(
    f"{model_checkpoint}-squad_scratch",  # name of the output folder and Hub repo
    eval_strategy="epoch",        # evaluate at the end of every epoch
    num_train_epochs=10,          # number of times to pass through the training data
    logging_strategy="steps",     # log training loss based on steps, not epochs
    logging_steps=10,             # log every 10 steps so we can watch loss drop
    learning_rate=2e-5,           # how big each weight update step is
    weight_decay=0.01,            # regularisation — penalises large weights to reduce overfitting
    push_to_hub=True,             # automatically upload the model to Hugging Face after training
)

# Wire everything together — model, training config, and datasets
trainer = Trainer(
    model=model,
    args=training_arg,
    train_dataset=small_sq_train_dataset,  # 100 shuffled training samples
    eval_dataset=small_sq_eval_dataset     # 100 shuffled validation samples
)

# Train the model and evaluate
# trainer.train() runs the full training loop
# trainer.evaluate() runs the model on the validation set and returns the loss
print("Starting training from scratch! Watch the loss metrics...")
trainer.train()

eval_results = trainer.evaluate()

# Perplexity = e^(eval_loss) — converts the raw loss into a more interpretable number
# Lower perplexity = the model is less surprised by the validation text = better
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Starting training from scratch! Watch the loss metrics...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,9.392185,8.124182
2,8.114401,7.526113
3,7.597704,6.544615
4,6.305912,5.912283
5,5.888002,5.599673
6,5.509960,5.426714
7,5.235763,5.319029
8,5.218224,5.256136
9,5.022217,5.222140
10,5.001159,5.210669


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Perplexity: 183.22


In [ ]:
# Load DistilGPT-2 WITH its pretrained weights this time
# Unlike from_config(), from_pretrained() loads the full model —
# architecture + weights already trained on a large corpus of internet text
# This model already understands grammar, vocabulary, and general language patterns
sq_pre_model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

# Same training configuration as the scratch experiment
# The only differences: a new output name and fewer epochs (3 instead of 10)
# Fewer epochs are needed because the model is already a strong starting point
training_arg = TrainingArguments(
    f"{model_checkpoint}-sq_gpt_pretrained",  # separate Hub repo from the scratch model
    eval_strategy="epoch",
    num_train_epochs=3,            # only 3 epochs — pretrained weights need less time to adapt
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

# Wire everything together with the pretrained model
trainer = Trainer(
    model=sq_pre_model,
    args=training_arg,
    train_dataset=small_sq_train_dataset,  # 100 training samples
    eval_dataset=small_sq_train_dataset    # 100 validation samples
)

# Train and evaluate — same process as before
# The difference in perplexity between this and the scratch model
# is the direct measure of how much pretrained knowledge is worth
print("Starting training from scratch! Watch the loss metrics...")
trainer.train()

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting training from scratch! Watch the loss metrics...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.668882,2.358156
2,2.450869,2.224475
3,2.339620,2.192337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Perplexity: 8.96


In [ ]:
# 1. Save the Scratch Model
model.push_to_hub(f"{model_checkpoint}-Squad-gpt-scratch-model")

# 2. Save the Pretrained Model
sq_pre_model.push_to_hub(f"{model_checkpoint}-Squad-gpt-pretrained-model")

# 3. Save the Tokenizer
tokenizer.push_to_hub(f"{model_checkpoint}-Squad-gpt-scratch-model")
tokenizer.push_to_hub(f"{model_checkpoint}-Squad-gpt-pretrained-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...boxfbzh/model.safetensors:  22%|##1       | 72.0MB /  328MB            

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...vp23qkk/model.safetensors:  34%|###4      |  112MB /  328MB            

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Ottneel/distilgpt2-Squad-gpt-pretrained-model/commit/f37c52d1524930116dd65f224e95f054425e00c9', commit_message='Upload tokenizer', commit_description='', oid='f37c52d1524930116dd65f224e95f054425e00c9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ottneel/distilgpt2-Squad-gpt-pretrained-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Ottneel/distilgpt2-Squad-gpt-pretrained-model'), pr_revision=None, pr_num=None)

In [4]:
# Load the fine-tuned SQuAD model back from Hugging Face Hub
# This is the model we just trained and pushed — we're pulling it back down
# to run inference and test how it generates questions
sq_test_model = AutoModelForCausalLM.from_pretrained("Ottneel/distilgpt2-Squad-gpt-pretrained-model")

# Load the matching tokenizer
# Must be the same tokenizer used during training so token IDs are consistent
tokenizer = AutoTokenizer.from_pretrained("Ottneel/distilgpt2-Squad-gpt-pretrained-model")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
# Context passage — a short excerpt from a tactical breakdown of Barcelona vs Newcastle
# This is out-of-distribution content: football analysis was not in SQuAD or TriviaQA
# We're using it to test how well the model generalises beyond its training data
start_text = """Newcastle came to Barcelona with a very specific, aggressive
game plan: a strict man-to-man press. Hansi Flick's solution was absolute,
chaotic fluidity. Barcelona registered just 6 successful dribbles all game,
bypassing the press with 63% possession, 401 accurate passes, and 10 Big Chances."""

# The prompt — we give the model the start of a question and let it complete it
prompt = "What did "

# Tokenize the context + prompt together as one input
# add_special_tokens=False — we don't want extra tokens interfering with generation
# return_tensors="pt" — return PyTorch tensors, which is what the model expects
inputs = tokenizer(
    start_text + prompt,
    add_special_tokens=False,
    return_tensors="pt"
)

outputs = sq_test_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],   # tells the model which tokens to attend to
    pad_token_id=tokenizer.eos_token_id,       # handles end-of-sequence cleanly
    max_length=inputs["input_ids"].shape[1] + 50,  # generate 50 new tokens beyond the input
    do_sample=True,                            # use probabilistic sampling, not greedy decoding
    top_k=50,                                  # only sample from the top 50 most likely tokens
    top_p=0.95,                                # nucleus sampling — covers 95% of probability mass
    temperature=0.5,                           # lower than default — more focused, less random
    repetition_penalty=1.3                     # penalises tokens that already appeared — prevents loops
)

# Decode the output token IDs back into readable text
# outputs[0] is the first (and only) generated sequence
print(tokenizer.decode(outputs[0]))

Newcastle came to Barcelona with a very specific, aggressive 
game plan: a strict man-to-man press. Hansi Flick's solution was absolute, 
chaotic fluidity. Barcelona registered just 6 successful dribbles all game, 
bypassing the press with 63% possession, 401 accurate passes, and 10 Big Chances.What did 
the player want? The ball is in his hands but he has been playing on it for over 20 years now! In this case you can see that there are no big differences between what they use as an attack or defensive midfield role - both of


**Trivial Dataset**

In [ ]:
dataset = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc.nocontext/train-00000-of-00001.parque(…):   0%|          | 0.00/55.4M [00:00<?, ?B/s]

rc.nocontext/validation-00000-of-00001.p(…):   0%|          | 0.00/7.34M [00:00<?, ?B/s]

rc.nocontext/test-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

In [ ]:
# Drop all columns except 'question'
# We only want the question text for training the question completion agent
# question_id, question_source — metadata we don't need
# entity_pages, search_results — supporting documents not relevant to our task
# answer — intentionally dropped because the goal is question generation, not answering
dataset = dataset.remove_columns([
    'question_id',
    'question_source',
    'entity_pages',
    'search_results',
    'answer'
])

# Apply the boundary marker to every question in the dataset
# Adds <|endoftext|> to the end of each question so the model knows
# where one question ends when everything gets concatenated later
dataset = dataset.map(add_end_of_text_to_question)

**Trivial Q&A Dataset**

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 138384
    })
    validation: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 17944
    })
    test: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 17210
    })
})

In [ ]:

# Define which model we're using — determines the tokenizer rules
model_checkpoint = "distilgpt2"

# Load the same tokenizer that was used when DistilGPT-2 was originally trained
# use_fast=True uses the Rust-based fast tokenizer for speed
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

# Define the tokenization function — converts raw question text into token IDs
# truncation=True ensures any question longer than the model's max length gets cut off
def tokenize_functions(examples):
    return tokenizer(examples["question"], truncation=True)

# Apply tokenization across the entire TriviaQA dataset
# batched=True processes multiple examples at once for efficiency
# num_proc=4 uses 4 CPU cores in parallel to speed things up
# remove_columns=['question'] drops the original text — now replaced by input_ids
# and attention_mask
tokenized_datasets = dataset.map(
    tokenize_functions,
    batched=True,
    num_proc=4,
    remove_columns=['question']
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map (num_proc=4):   0%|          | 0/138384 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/17944 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/17210 [00:00<?, ? examples/s]

In [ ]:
block_size = 128
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4
)



Map (num_proc=4):   0%|          | 0/138384 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/17944 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/17210 [00:00<?, ? examples/s]

In [ ]:
# Shrink dataset for faster demo
small_train_dataset = lm_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = lm_datasets["validation"].shuffle(seed=42).select(range(100))

**Trivial Model Training from scratch**

In [ ]:
# Model Training from scratch
notebook_login()
# Step A: Get the blueprint for the model
print("Fetching model blueprint...")
config = AutoConfig.from_pretrained(model_checkpoint)

# Step B: Build a blank model using that blueprint
print("Building a blank model from scratch based on the Trivial Dataset...")
model = AutoModelForCausalLM.from_config(config)

training_arg = TrainingArguments(
    f"{model_checkpoint}-trivial_scratch",
    eval_strategy="epoch",
    num_train_epochs=3,
    logging_strategy="steps",  # Tell it to log based on steps
    logging_steps=10,          # Log the training loss every 10 steps
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

Fetching model blueprint...
Building a blank model from scratch...


In [ ]:
trainer = Trainer(
    model=model,
    args=training_arg,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset
)


# 6. TRAIN & EVALUATE

print("Starting training from scratch! Watch the loss metrics...")
trainer.train()

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Starting training from scratch! Watch the loss metrics...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,9.834692,8.764684
2,8.820914,8.394892
3,8.572545,8.285946


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Perplexity: 3967.72


**Fine tuning a pretrained model on the trivial Dataset**

In [ ]:
pre_model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
training_arg = TrainingArguments(
    f"{model_checkpoint}-trivial_gptpretrained",
    eval_strategy="epoch",
    num_train_epochs=3,
    logging_strategy="steps",  # Tell it to log based on steps
    logging_steps=10,          # Log the training loss every 10 steps
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
trainer = Trainer(
    model=pre_model,
    args=training_arg,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset
)


# 6. TRAIN & EVALUATE

print("Starting pretrained model. Watch the loss metrics...")
trainer.train()

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Starting training from scratch! Watch the loss metrics...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.568643,3.219791
2,3.219341,3.082724
3,3.188135,3.052274


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Perplexity: 21.16


In [ ]:
pre_model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
training_arg = TrainingArguments(
    f"{model_checkpoint}-trivial_gptpretrained",
    eval_strategy="epoch",
    num_train_epochs=8,
    logging_strategy="steps",  # Tell it to log based on steps
    logging_steps=10,          # Log the training loss every 10 steps
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=pre_model,
    args=training_arg,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset
)


# 6. TRAIN & EVALUATE

print("Starting pretrained model. Watch the loss metrics...")
trainer.train()

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting pretrained model. Watch the loss metrics...


Epoch,Training Loss,Validation Loss
1,3.562272,3.194006
2,3.181017,3.021676
3,3.110596,2.959125
4,2.870511,2.926252
5,2.818984,2.903171
6,2.764508,2.891969
7,2.744557,2.885138
8,2.691495,2.883301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Perplexity: 17.87


In [ ]:
# 1. Save the Scratch Model
model.push_to_hub(f"{model_checkpoint}-trivial-gpt-scratch-model")

# 2. Save the Pretrained Model
pre_model.push_to_hub(f"{model_checkpoint}-trivial-gpt-pretrained-model")

# 3. Save the Tokenizer
tokenizer.push_to_hub(f"{model_checkpoint}-trivial-gpt-scratch-model")
tokenizer.push_to_hub(f"{model_checkpoint}-trivial-gpt-pretrained-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hyg0b4n/model.safetensors:  37%|###6      |  120MB /  328MB            

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...od3dpea/model.safetensors:  34%|###4      |  112MB /  328MB            

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Ottneel/distilgpt2-trivial-gpt-pretrained-model/commit/d40fdde83a607ed40dcba31dfcc62bf9aec009e7', commit_message='Upload tokenizer', commit_description='', oid='d40fdde83a607ed40dcba31dfcc62bf9aec009e7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ottneel/distilgpt2-trivial-gpt-pretrained-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Ottneel/distilgpt2-trivial-gpt-pretrained-model'), pr_revision=None, pr_num=None)